In [0]:
-- create or replace view finances.03_gold.all_transactions as
  with statements as (
    select 
      *,
      get(split(replace(Description,'*',' '), ' '), 0) as FirstOne,
      concat_ws(' ', get(split(replace(Description,'*',' '), ' '), 0), get(split(Description, ' '), 1)) as FirstTwo
    from (
      select Account, Date, Code, upper(regexp_replace(Description, ' +', ' ')) as Description, EntryIs, Amount from finances.02_silver.bank_statements
      union all
      select Account, Date, null, upper(regexp_replace(Description, ' +', ' ')) as Description, EntryIs, Amount from finances.02_silver.creditcard_statements
    ) t
  )

  select
    s.Account,
    s.Date,
    coalesce(coalesce(coalesce(coalesce(d.AltDesc,c.AltDesc), f1.AltDesc), f2.AltDesc), s.Description) as Description,
    s.EntryIs,
    coalesce(coalesce(coalesce(d.Category, c.Category), f1.Category), f2.Category) as BudgetCategory,
    Amount
  from statements s
  left join finances.refdata.description__category  d on  d.Field='Description' and  d.Description=s.Description
  left join finances.refdata.description__category  c on  c.Field='Code'        and  c.Description=s.Code
  left join finances.refdata.description__category f1 on f1.Field='FirstOne'    and f1.Description=s.FirstOne
  left join finances.refdata.description__category f2 on f2.Field='FirstTwo'    and f2.Description=s.FirstTwo


